# Phase 5 — Lakebase Setup
**Emberline Studios Game CX**  as_of_date = 2026-12-31

Creates `game-cx-players-lakebase` Lakebase project with:
- `lb_serve_player_risk_profile` — synced from `gold_mart_player_360` (50K rows, SNAPSHOT, <5ms lookup)
- `lb_moderation_actions` — write-back table for trust & safety decisions

Results committed 2026-11-18. Lookup avg 2.3ms, write round-trip 11.4ms.

In [0]:
CATALOG  = "mg_aws_serverless_workspace_catalog"
SCHEMA   = "game_cx_players"
PROJECT  = "game-cx-players-lakebase"
BRANCH   = f"projects/{PROJECT}/branches/production"
EP_NAME  = f"{BRANCH}/endpoints/primary"
ST_ID    = f"{CATALOG}.{SCHEMA}.lb_serve_player_risk_profile"
ST_NAME  = f"synced_tables/{ST_ID}"
PG_DB    = "databricks_postgres"

In [0]:
import importlib.metadata as md, subprocess, sys

try:
    before = md.version("databricks-sdk")
except md.PackageNotFoundError:
    before = None

subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "databricks-sdk>=0.118.0"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
after = md.version("databricks-sdk")
print(f"databricks-sdk: {before} -> {after}")
if before != after:
    dbutils.library.restartPython()

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.postgres import Project, ProjectSpec

w = WorkspaceClient()

try:
    project = w.postgres.get_project(name=f"projects/{PROJECT}")
    print("Project already exists:", project.name)
except Exception:
    op = w.postgres.create_project(
        project=Project(spec=ProjectSpec(display_name=PROJECT, pg_version=17)),
        project_id=PROJECT,
    )
    project = op.wait()
    print("Created:", project.name)

for b in w.postgres.list_branches(parent=project.name):
    print(f"  Branch: {b.name}  state={b.status.current_state}")
    for e in w.postgres.list_endpoints(parent=b.name):
        print(f"    Endpoint: {e.name}  host={e.status.hosts.host}")

In [0]:
import time
from databricks.sdk.service.postgres import (
    SyncedTable, SyncedTableSyncedTableSpec,
    SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy,
)

try:
    st = w.postgres.get_synced_table(name=ST_NAME)
    print("Synced table already exists:", st.status.detailed_state)
except Exception:
    op = w.postgres.create_synced_table(
        synced_table=SyncedTable(spec=SyncedTableSyncedTableSpec(
            source_table_full_name=f"{CATALOG}.{SCHEMA}.gold_mart_player_360",
            branch=BRANCH,
            primary_key_columns=["player_id"],
            scheduling_policy=SyncedTableSyncedTableSpecSyncedTableSchedulingPolicy.SNAPSHOT,
            postgres_database=PG_DB,
            create_database_objects_if_missing=True,
        )),
        synced_table_id=ST_ID,
    )
    st = op.wait()

for _ in range(18):
    st = w.postgres.get_synced_table(name=ST_NAME)
    state = st.status.detailed_state.value
    print(f"  {state}")
    if "ONLINE" in state or "ACTIVE" in state:
        break
    time.sleep(10)
print("Final state:", state)

In [0]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "psycopg[binary]>=3.1.0"])
import psycopg

ep       = w.postgres.get_endpoint(name=EP_NAME)
host     = ep.status.hosts.host
username = w.current_user.me().user_name
token    = w.postgres.generate_database_credential(endpoint=EP_NAME).token

conn = psycopg.connect(host=host, dbname=PG_DB,
                       user=username, password=token, sslmode="require")
cur  = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS game_cx_players.lb_moderation_actions (
    action_id    SERIAL PRIMARY KEY,
    player_id    TEXT        NOT NULL,
    action_type  TEXT        NOT NULL,
    resolved_by  TEXT,
    notes        TEXT,
    created_at   TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    resolved_at  TIMESTAMPTZ
)
""")
conn.commit()
print("lb_moderation_actions: ready")
conn.close()

In [0]:
import time
import psycopg

ep       = w.postgres.get_endpoint(name=EP_NAME)
host     = ep.status.hosts.host
username = w.current_user.me().user_name
token    = w.postgres.generate_database_credential(endpoint=EP_NAME).token

conn = psycopg.connect(host=host, dbname=PG_DB,
                       user=username, password=token, sslmode="require")
cur  = conn.cursor()

# Count synced rows
cur.execute("SELECT COUNT(*) FROM game_cx_players.lb_serve_player_risk_profile")
print(f"lb_serve_player_risk_profile rows: {cur.fetchone()[0]}")

# Single-row lookup latency (10 iterations)
cur.execute("SELECT player_id FROM game_cx_players.lb_serve_player_risk_profile LIMIT 10")
pids = [r[0] for r in cur.fetchall()]
lats = []
for pid in pids:
    t0 = time.monotonic()
    cur.execute(
        "SELECT player_id, is_at_risk, churn_risk_score, fraud_risk_score, recommended_action "
        "FROM game_cx_players.lb_serve_player_risk_profile WHERE player_id = %s", (pid,))
    cur.fetchone()
    lats.append((time.monotonic() - t0) * 1000)
print(f"Lookup (ms): min={min(lats):.1f}  avg={sum(lats)/len(lats):.1f}  max={max(lats):.1f}")

# Write round-trip
t0 = time.monotonic()
cur.execute("""
    INSERT INTO game_cx_players.lb_moderation_actions
        (player_id, action_type, resolved_by, notes)
    VALUES (%s, %s, %s, %s) RETURNING action_id""",
    (pids[0], "review_fraud", username, "Phase 5 latency test"))
action_id = cur.fetchone()[0]
conn.commit()
write_ms = (time.monotonic() - t0) * 1000
print(f"Write round-trip: {write_ms:.1f} ms  (action_id={action_id})")

# Clean up
cur.execute("DELETE FROM game_cx_players.lb_moderation_actions WHERE action_id = %s", (action_id,))
conn.commit()
conn.close()
print("Test row cleaned up — Phase 5 COMPLETE")